In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,f1_score, roc_auc_score)
import time


In [3]:
!git clone https://github.com/ZiadXI/Brazilian-E-Commerce.git

fatal: destination path 'Brazilian-E-Commerce' already exists and is not an empty directory.


In [4]:
# 1. Load the cleaned dataset
df = pd.read_csv('Brazilian-E-Commerce/data/clean_orders/orders.csv')
# 2. Convert dates to datetime to calculate 'expected_days'
df['order_date'] = pd.to_datetime(df['order_date'])
df['expected_delivery_date'] = pd.to_datetime(df['expected_delivery_date'])
df['expected_days'] = (df['expected_delivery_date'] - df['order_date']).dt.days

# 3. NO LEAKAGE RULE: Select ONLY the allowed features + target
# We specifically do NOT include actual_delivery_days, delivery_status, delivery_delay, etc.
allowed_columns = [
    'courier', 'expected_days', 'weather', 'season',
    'area', 'category', 'order_month', 'order_hour',
    'is_late' # Target
]

# Create our strict modeling dataframe and drop any rows with NaN in these specific columns
model_df = df[allowed_columns].dropna().copy()

print(f"Dataset shape ready for ML: {model_df.shape}")


Dataset shape ready for ML: (113314, 9)


In [5]:
# 1. Define X (Features) and y (Target)
X = model_df.drop(columns=['is_late'])
y = model_df['is_late']

# 2. Train/Test Split (80/20) - Stratified ensures same % of late orders in both sets!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Identify categorical vs numerical columns
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

# 4. Preprocessing: Scale numbers, One-Hot Encode categories
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# Fit on training data, transform both
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# Get the new column names after OneHotEncoding
encoded_cat_cols = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(encoded_cat_cols)

print(f"Number of features after encoding: {X_train_encoded.shape[1]}")


Number of features after encoding: 114


In [6]:
print("Train shape:", X_train_encoded.shape)
print("Test shape:", X_test_encoded.shape)
print("Target distribution (train):")
print(y_train.value_counts(normalize=True))

Train shape: (90651, 114)
Test shape: (22663, 114)
Target distribution (train):
is_late
0    0.935632
1    0.064368
Name: proportion, dtype: float64


# Random Forest - HyperParameter Tuning

In [7]:
print("Tuning Random Forest...")
t0 = time.time()

rf_param_dist = {
    'n_estimators': [100, 150, 200],
    'max_depth': [10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', None]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_param_dist, n_iter=10, cv=3, scoring='f1',
    random_state=42, n_jobs=-1
)
rf_search.fit(X_train_encoded, y_train)
rf_best = rf_search.best_estimator_

print("RF best params:", rf_search.best_params_)
print(f"Tuning time: {time.time()-t0:.1f}s")

Tuning Random Forest...
RF best params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 15, 'class_weight': 'balanced'}
Tuning time: 778.9s


In [12]:
rf_best = rf_search.best_estimator_
import joblib
joblib.dump(rf_best, 'rf_best_model.pkl')

['rf_best_model.pkl']

# Random Forset Evaluation Metrics

In [9]:
rf_pred = rf_best.predict(X_test_encoded)
rf_proba = rf_best.predict_proba(X_test_encoded)[:, 1]

rf_results = {
    'Accuracy':  accuracy_score(y_test, rf_pred),
    'Precision': precision_score(y_test, rf_pred),
    'Recall':    recall_score(y_test, rf_pred),
    'F1':        f1_score(y_test, rf_pred),
    'ROC-AUC':   roc_auc_score(y_test, rf_proba),
}
print("Random Forest Results :")
for k, v in rf_results.items():
    print(f"{k}: {v:.4f}")

Random Forest Results :
Accuracy: 0.8226
Precision: 0.1810
Recall: 0.4983
F1: 0.2656
ROC-AUC: 0.7533


he Random Forest model detects approximately half of the actual late deliveries (Recall = 49.83%), meaning it can identify around 50% of orders that will actually be late. However, its low Precision (18.10%) indicates a high number of false alarms: among orders predicted as late, only about 18% are actually late. This represents a trade-off where the model prioritizes detecting more late deliveries, which may be related to using class_weight='balanced'. However, the impact of balancing should be confirmed by comparing it with a model using class_weight=None.

# Random Forest Balanced VS. No Weight

In [10]:
params_no_weight = {
    k: v for k, v in rf_search.best_params_.items()
    if k != 'class_weight'
}

params_no_weight['class_weight'] = None

rf_no_weight = RandomForestClassifier(
    **params_no_weight,
    random_state=42,
    n_jobs=-1
)

rf_no_weight.fit(X_train_encoded, y_train)

pred_nw = rf_no_weight.predict(X_test_encoded)
proba_nw = rf_no_weight.predict_proba(X_test_encoded)[:, 1]

rf_no_weight_results = {
    'Accuracy': accuracy_score(y_test, pred_nw),
    'Precision': precision_score(y_test, pred_nw, zero_division=0),
    'Recall': recall_score(y_test, pred_nw, zero_division=0),
    'F1': f1_score(y_test, pred_nw, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, proba_nw)
}

print("\n Random Forest (Without class_weight) - Comparison :")

for k, v in rf_no_weight_results.items():
    print(f"{k}: {v:.4f}")


 Random Forest (Without class_weight) - Comparison :
Accuracy: 0.9356
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
ROC-AUC: 0.7526


Without class weighting, the Random Forest achieved a higher Accuracy (93.56%), but completely failed to identify late deliveries (Recall = 0%, F1 = 0%). In contrast, using class_weight='balanced' reduced Accuracy to 82.26% but substantially improved the detection of late deliveries, achieving 49.83% Recall and 26.56% F1. Therefore, class_weight='balanced' is more appropriate for this imbalanced classification problem because the main goal is to detect late deliveries rather than simply maximize overall Accuracy.

# KNN - Hyperparameter Tuning & Final Training

In [15]:
from sklearn.model_selection import train_test_split as tts
from sklearn.decomposition import TruncatedSVD

# Use a stratified subsample for hyperparameter tuning
X_knn_tune, _, y_knn_tune, _ = tts(
    X_train_encoded,
    y_train,
    train_size=20000,
    random_state=42,
    stratify=y_train
)

print("Tuning KNN...")
t0 = time.time()

knn_param_dist = {
    'n_neighbors': [5, 9, 15, 21, 31],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_search = RandomizedSearchCV(
    KNeighborsClassifier(n_jobs=-1),
    knn_param_dist,
    n_iter=8,
    cv=3,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

knn_search.fit(X_knn_tune, y_knn_tune)

print("KNN best params:", knn_search.best_params_)
print(f"Tuning time: {time.time() - t0:.1f}s")

svd = TruncatedSVD(n_components=100, random_state=42)
X_train_reduced = svd.fit_transform(X_train_encoded)

print(f"Explained variance ratio (sum): {svd.explained_variance_ratio_.sum():.4f}")

# Train the final KNN model on the full training set (reduced dimensions)
knn_best = KNeighborsClassifier(
    **knn_search.best_params_,
    n_jobs=-1
)

knn_best.fit(X_train_reduced, y_train)

print("Final KNN trained on the full training set.")

Tuning KNN...
KNN best params: {'weights': 'distance', 'n_neighbors': 5, 'metric': 'manhattan'}
Tuning time: 153.1s
Explained variance ratio (sum): 0.9997
Final KNN trained on the full training set.


In [17]:
joblib.dump(knn_best, 'knn_best_model.pkl')

['knn_best_model.pkl']

# KNN - Evaluation

In [16]:
print("Evaluating KNN :")

X_test_reduced = svd.transform(X_test_encoded)

knn_proba = knn_best.predict_proba(X_test_reduced)[:, 1]

knn_pred = (knn_proba >= 0.5).astype(int)

knn_results = {
    'Accuracy': accuracy_score(y_test, knn_pred),
    'Precision': precision_score(y_test, knn_pred, zero_division=0),
    'Recall': recall_score(y_test, knn_pred, zero_division=0),
    'F1': f1_score(y_test, knn_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, knn_proba)
}

print("KNN Results:")

for k, v in knn_results.items():
    print(f"{k}: {v:.4f}")

Evaluating KNN :
KNN Results:
Accuracy: 0.9302
Precision: 0.2786
Recall: 0.0535
F1: 0.0897
ROC-AUC: 0.6056


In [ ]:
import numpy as np

print("Trying different thresholds for Random Forest:\n")

rf_proba = rf_best.predict_proba(X_test_encoded)[:, 1]

best_f1 = 0
best_threshold = 0.5

for threshold in np.arange(0.2, 0.6, 0.05):
    pred_t = (rf_proba >= threshold).astype(int)

    acc = accuracy_score(y_test, pred_t)
    prec = precision_score(y_test, pred_t, zero_division=0)
    rec = recall_score(y_test, pred_t, zero_division=0)
    f1 = f1_score(y_test, pred_t, zero_division=0)

    print(f"Threshold = {threshold:.2f} | Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"\nBest threshold: {best_threshold:.2f} with F1: {best_f1:.4f}")